In [ ]:
import numpy as np
from finesse.cymath.complex import cnorm
from finesse.detectors.workspace import DetectorWorkspace

import finesse
from finesse.detectors import Detector
from finesse.script.spec import KATSPEC, make_element


class VegetaWorkspace(DetectorWorkspace):
    def __init__(self, owner, sim):
        super().__init__(owner, sim, needs_carrier=True)
        self.set_output_fn(calculate_vegeta_power)


def calculate_vegeta_power(dws: DetectorWorkspace):
    field_vector = dws.sim.carrier.node_field_vector(dws.owner.node, freq_idx=0)
    BASE_POWER = 9000
    return (cnorm(field_vector[0]) * 0.5) + BASE_POWER


class VegetaDetector(Detector):
    def __init__(
        self,
        name,
        node,
    ):
        super().__init__(name, node, dtype=np.float64, unit="W", label="Power")

    def _get_workspace(self, sim):
        return VegetaWorkspace(self, sim)


KATSPEC.register_element(make_element(VegetaDetector, "vegeta"))

model = finesse.Model("""
l l1 P=1
vegeta super_power l1.p1.o
""")
sol = model.run()
sol["super_power"]

np.float64(9001.0)